# Testing AzureML Models from Databricks

Comprehensive guide for testing and validating AzureML deployed models using Databricks as the testing client.

## Testing Scenarios:
1. Quick single-record test
2. REST API testing
3. Azure ML SDK approach
4. Batch testing with Pandas
5. Distributed scoring with Pandas UDF
6. Performance and latency testing
7. Error handling and validation

## Setup & Configuration

In [ ]:
# Import required libraries
import os
import json
import requests
import pandas as pd
import numpy as np
import time
import logging
from datetime import datetime
from typing import Dict, List, Any

# Latest Azure ML SDK v2
from azure.identity import DefaultAzureCredential, ClientSecretCredential, EnvironmentCredential, ChainedTokenCredential
from azure.ai.ml import MLClient
from azure.core.exceptions import HttpResponseError
print("✓ Azure ML SDK v2 (azure-ai-ml) imported successfully")

# Latest Databricks SDK (databricks-sdk)
try:
    from databricks.sdk import WorkspaceClient
    print("✓ Databricks SDK (databricks-sdk) imported successfully")
except ImportError:
    print("Note: databricks-sdk not installed. Install with: pip install databricks-sdk")
    WorkspaceClient = None

# Pyspark
from pyspark.sql import functions as F
from pyspark.sql.functions import pandas_udf
from pyspark.sql.types import DoubleType, StringType

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(name)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

### Configure Environment Variables

In [ ]:
# AzureML Configuration
SUBSCRIPTION_ID = os.getenv("AZURE_SUBSCRIPTION_ID", "")
RESOURCE_GROUP = os.getenv("AZURE_RESOURCE_GROUP", "")
WORKSPACE_NAME = os.getenv("AZUREML_WORKSPACE_NAME", "")
ENDPOINT_URL = os.getenv("AZUREML_ENDPOINT_URL", "")

# Azure Authentication
TENANT_ID = os.getenv("AZURE_TENANT_ID", "")
CLIENT_ID = os.getenv("AZURE_CLIENT_ID", "")
CLIENT_SECRET = os.getenv("AZURE_CLIENT_SECRET", "")

# Optional: Model endpoint name (if using SDK)
ENDPOINT_NAME = dbutils.widgets.get("endpoint_name", "")

print("Configuration Summary:")
print(f"  Subscription: {SUBSCRIPTION_ID[:20]}..." if SUBSCRIPTION_ID else "  Subscription: Not set")
print(f"  Resource Group: {RESOURCE_GROUP}")
print(f"  Workspace: {WORKSPACE_NAME}")
print(f"  Endpoint URL: {ENDPOINT_URL[:50]}..." if ENDPOINT_URL else "  Endpoint URL: Not set")

### Initialize Authentication

In [ ]:
# Get authentication credential
if all([TENANT_ID, CLIENT_ID, CLIENT_SECRET]):
    credential = ClientSecretCredential(
        tenant_id=TENANT_ID,
        client_id=CLIENT_ID,
        client_secret=CLIENT_SECRET
    )
    auth_method = "Service Principal"
else:
    credential = DefaultAzureCredential(exclude_interactive_browser_credential=True)
    auth_method = "Managed Identity"

print(f"✓ Authenticated using: {auth_method}")

# Get token for REST API calls
token = credential.get_token("https://ml.azure.com/.default").token
print(f"✓ Token acquired (expires in ~1 hour)")

## Test 1: Simple Single-Record Test

In [ ]:
# Define test data (single record)
test_sample = {
    "input_data": {
        "columns": ["feature1", "feature2", "feature3", "feature4"],
        "data": [[1.5, 2.3, 3.1, 4.2]]  # Single sample
    }
}

print("Test Data:")
print(json.dumps(test_sample, indent=2))

# Prepare request
headers = {
    "Authorization": f"Bearer {token}",
    "Content-Type": "application/json"
}

# Score
print("\nCalling endpoint...")
try:
    response = requests.post(
        ENDPOINT_URL,
        headers=headers,
        json=test_sample,
        timeout=30
    )
    
    print(f"\nResponse Status: {response.status_code}")
    
    if response.status_code == 200:
        result = response.json()
        print(f"✓ Prediction successful")
        print(f"Result: {result}")
    else:
        print(f"✗ Error: {response.status_code}")
        print(f"Response: {response.text}")
        
except Exception as e:
    print(f"✗ Request failed: {e}")

## Test 2: Multiple Records in Batch

In [ ]:
# Test with multiple records
test_batch = {
    "input_data": {
        "columns": ["feature1", "feature2", "feature3", "feature4"],
        "data": [
            [1.5, 2.3, 3.1, 4.2],
            [2.0, 2.5, 3.5, 4.5],
            [1.0, 2.0, 3.0, 4.0],
            [3.0, 3.5, 4.0, 5.0],
            [2.5, 3.0, 3.5, 4.5]
        ]
    }
}

print(f"Testing batch of {len(test_batch['input_data']['data'])} records...\n")

try:
    response = requests.post(
        ENDPOINT_URL,
        headers=headers,
        json=test_batch,
        timeout=60
    )
    
    if response.status_code == 200:
        predictions = response.json()
        
        # Format results
        results_df = pd.DataFrame({
            "feature1": [x[0] for x in test_batch["input_data"]["data"]],
            "feature2": [x[1] for x in test_batch["input_data"]["data"]],
            "prediction": predictions if isinstance(predictions, list) else [predictions] * len(test_batch["input_data"]["data"])
        })
        
        print("Batch Predictions:")
        display(results_df)
        
        # Statistics
        print(f"\nStatistics:")
        print(f"  Mean: {np.mean(results_df['prediction']):.4f}")
        print(f"  Std Dev: {np.std(results_df['prediction']):.4f}")
        print(f"  Min: {np.min(results_df['prediction']):.4f}")
        print(f"  Max: {np.max(results_df['prediction']):.4f}")
    else:
        print(f"Error: {response.status_code}")
        print(response.text)
        
except Exception as e:
    print(f"Request failed: {e}")

## Test 3: Testing with Azure ML SDK

In [ ]:
# Initialize ML Client
try:
    ml_client = MLClient(
        credential=credential,
        subscription_id=SUBSCRIPTION_ID,
        resource_group_name=RESOURCE_GROUP,
        workspace_name=WORKSPACE_NAME
    )
    
    print(f"✓ Connected to AzureML workspace: {WORKSPACE_NAME}")
    
    # List available endpoints
    endpoints = ml_client.online_endpoints.list()
    endpoint_list = list(endpoints)
    
    print(f"\nAvailable Endpoints ({len(endpoint_list)}):")
    for ep in endpoint_list[:10]:  # Show first 10
        print(f"  - {ep.name} ({ep.provisioning_state})")
        
except Exception as e:
    print(f"Note: SDK connection requires proper workspace configuration: {e}")

## Test 4: Get Endpoint Details via SDK

In [ ]:
# Get specific endpoint details
if ENDPOINT_NAME:
    try:
        endpoint = ml_client.online_endpoints.get(ENDPOINT_NAME)
        
        print(f"Endpoint: {endpoint.name}")
        print(f"Status: {endpoint.provisioning_state}")
        print(f"Auth Mode: {endpoint.auth_mode}")
        print(f"Scoring URI: {endpoint.scoring_uri}")
        print(f"Deployments: {len(endpoint.deployments) if endpoint.deployments else 0}")
        
        if endpoint.deployments:
            print(f"\nDeployments:")
            for dep_name in endpoint.deployments:
                deploy = ml_client.online_deployments.get(
                    deployment_name=dep_name,
                    endpoint_name=ENDPOINT_NAME
                )
                print(f"  - {deploy.name}")
                print(f"    Model: {deploy.model}")
                print(f"    Compute: {deploy.compute}")
                print(f"    Instance Count: {deploy.instance_count}")
                
    except Exception as e:
        print(f"Could not retrieve endpoint details: {e}")
else:
    print("Endpoint name not provided. Use 'endpoint_name' widget to specify.")

## Test 5: Load Test Data & Test with Pandas DataFrame

In [ ]:
# Create sample test data
np.random.seed(42)
n_samples = 20

test_data = pd.DataFrame({
    'feature1': np.random.uniform(0, 5, n_samples),
    'feature2': np.random.uniform(0, 5, n_samples),
    'feature3': np.random.uniform(0, 5, n_samples),
    'feature4': np.random.uniform(0, 5, n_samples)
})

print(f"Test Data ({n_samples} records):")
display(test_data.head(5))

In [ ]:
# Score using Pandas
feature_cols = ['feature1', 'feature2', 'feature3', 'feature4']

# Prepare payload
payload = {
    "input_data": {
        "columns": feature_cols,
        "data": test_data[feature_cols].values.tolist()
    }
}

print(f"Scoring {len(test_data)} records...")

try:
    response = requests.post(
        ENDPOINT_URL,
        headers=headers,
        json=payload,
        timeout=60
    )
    
    if response.status_code == 200:
        predictions = response.json()
        
        # Create results dataframe
        test_data['prediction'] = predictions if isinstance(predictions, list) else [predictions] * len(test_data)
        test_data['prediction_time'] = datetime.now()
        
        print(f"✓ Scoring completed successfully")
        print(f"\nResults:")
        display(test_data[['feature1', 'feature2', 'prediction']].head(10))
        
        # Summary statistics
        print(f"\nPrediction Summary:")
        print(test_data['prediction'].describe())
    else:
        print(f"Error: {response.status_code}")
        print(response.text)
        
except Exception as e:
    print(f"Scoring failed: {e}")

## Test 6: Distributed Scoring with Pandas UDF

# Refresh token for distributed execution
token = credential.get_token("https://ml.azure.com/.default").token

@pandas_udf("double")
def score_batch_udf(feature_cols_iter):
    """
    Pandas UDF for distributed batch scoring.
    Processes batches in parallel across Spark workers.
    """
    for feature_batch in feature_cols_iter:
        # Prepare batch payload
        payload = {
            "input_data": {
                "columns": ['feature1', 'feature2', 'feature3', 'feature4'],
                "data": feature_batch[['feature1', 'feature2', 'feature3', 'feature4']].values.tolist()
            }
        }
        
        headers = {
            "Authorization": f"Bearer {token}",
            "Content-Type": "application/json"
        }
        
        try:
            response = requests.post(
                ENDPOINT_URL,
                headers=headers,
                json=payload,
                timeout=60
            )
            
            if response.status_code == 200:
                predictions = response.json()
                # Return as pandas Series
                yield pd.Series(predictions if isinstance(predictions, list) else [predictions] * len(feature_batch))
            else:
                # Return NaN on error
                yield pd.Series([float('nan')] * len(feature_batch))
        except Exception as e:
            # Return NaN on exception
            yield pd.Series([float('nan')] * len(feature_batch))

# Apply UDF to Spark DataFrame
print("Applying distributed scoring UDF...\n")
df_scored = df_spark.withColumn(
    "prediction",
    score_batch_udf(
        F.struct('feature1', 'feature2', 'feature3', 'feature4')
    )
)

# Trigger computation
print(f"Scored {df_scored.count()} rows")
print("\nResults:")
display(df_scored.select('feature1', 'feature2', 'prediction').limit(10))

# Latency test: Single request
print("Single Record Latency Test\n")
print("Sending 10 requests...\n")

latencies = []
successful = 0
failed = 0

test_payload = {
    "input_data": {
        "columns": ["feature1", "feature2", "feature3", "feature4"],
        "data": [[1.0, 2.0, 3.0, 4.0]]
    }
}

for i in range(10):
    start_time = time.time()
    
    try:
        response = requests.post(
            ENDPOINT_URL,
            headers=headers,
            json=test_payload,
            timeout=30
        )
        
        elapsed = (time.time() - start_time) * 1000  # Convert to milliseconds
        latencies.append(elapsed)
        
        status = "✓" if response.status_code == 200 else "✗"
        print(f"Request {i+1}: {elapsed:6.2f}ms {status} (HTTP {response.status_code})")
        
        if response.status_code == 200:
            successful += 1
        else:
            failed += 1
    except Exception as e:
        elapsed = (time.time() - start_time) * 1000
        latencies.append(elapsed)
        print(f"Request {i+1}: {elapsed:6.2f}ms ✗ (Error: {str(e)[:30]})")
        failed += 1

# Summary statistics
print(f"\nLatency Summary:")
print(f"  Total Requests: {len(latencies)}")
print(f"  Successful: {successful}")
print(f"  Failed: {failed}")
print(f"  Min: {min(latencies):.2f}ms")
print(f"  Max: {max(latencies):.2f}ms")
print(f"  Mean: {np.mean(latencies):.2f}ms")
print(f"  Std Dev: {np.std(latencies):.2f}ms")
print(f"  Median: {np.median(latencies):.2f}ms")
print(f"  p95: {np.percentile(latencies, 95):.2f}ms")
print(f"  p99: {np.percentile(latencies, 99):.2f}ms")

# Test various error scenarios
print("Testing Error Scenarios\n")

# Test 1: Missing required columns
print("1. Missing required column:")
try:
    bad_payload = {
        "input_data": {
            "columns": ["feature1", "feature2"],  # Missing feature3, feature4
            "data": [[1.0, 2.0]]
        }
    }
    response = requests.post(ENDPOINT_URL, headers=headers, json=bad_payload, timeout=10)
    print(f"   Status: {response.status_code}")
    if response.status_code != 200:
        print(f"   Error: {response.text[:100]}")
except Exception as e:
    print(f"   Error: {e}")

# Test 2: Invalid data type
print("\n2. Invalid data type:")
try:
    bad_payload = {
        "input_data": {
            "columns": ["feature1", "feature2", "feature3", "feature4"],
            "data": [["invalid", "string", "values", "here"]]
        }
    }
    response = requests.post(ENDPOINT_URL, headers=headers, json=bad_payload, timeout=10)
    print(f"   Status: {response.status_code}")
    if response.status_code != 200:
        print(f"   Error: {response.text[:100]}")
except Exception as e:
    print(f"   Error: {e}")

# Test 3: Empty data
print("\n3. Empty data:")
try:
    bad_payload = {
        "input_data": {
            "columns": ["feature1", "feature2", "feature3", "feature4"],
            "data": []
        }
    }
    response = requests.post(ENDPOINT_URL, headers=headers, json=bad_payload, timeout=10)
    print(f"   Status: {response.status_code}")
    if response.status_code != 200:
        print(f"   Error: {response.text[:100]}")
except Exception as e:
    print(f"   Error: {e}")

# Test 4: Invalid endpoint URL
print("\n4. Invalid endpoint URL:")
try:
    bad_url = "https://invalid-endpoint.eastus.inference.ml.azure.com/score"
    response = requests.post(bad_url, headers=headers, json=test_payload, timeout=10)
    print(f"   Status: {response.status_code}")
except Exception as e:
    print(f"   Expected error: {type(e).__name__}")

def validate_model_response(response: Dict[str, Any], test_size: int) -> Dict[str, Any]:
    """
    Comprehensive validation of model response.
    """
    validation_results = {
        "http_status": response.get("status_code"),
        "response_format_valid": False,
        "prediction_count_matches": False,
        "prediction_values_valid": False,
        "issues": []
    }
    
    try:
        # Check if predictions exist
        predictions = response.get("predictions")
        
        if predictions is None:
            validation_results["issues"].append("No predictions in response")
            return validation_results
        
        validation_results["response_format_valid"] = True
        
        # Check count
        pred_list = predictions if isinstance(predictions, list) else [predictions]
        
        if len(pred_list) != test_size:
            validation_results["issues"].append(
                f"Prediction count mismatch: expected {test_size}, got {len(pred_list)}"
            )
        else:
            validation_results["prediction_count_matches"] = True
        
        # Check for NaN/Inf
        has_nan = any(pd.isna(p) for p in pred_list)
        has_inf = any(np.isinf(p) if isinstance(p, (int, float)) else False for p in pred_list)
        
        if has_nan:
            validation_results["issues"].append("Predictions contain NaN values")
        if has_inf:
            validation_results["issues"].append("Predictions contain Inf values")
        
        if not (has_nan or has_inf):
            validation_results["prediction_values_valid"] = True
        
    except Exception as e:
        validation_results["issues"].append(f"Validation error: {str(e)}")
    
    return validation_results

# Test the validation function
test_payload = {
    "input_data": {
        "columns": ["feature1", "feature2", "feature3", "feature4"],
        "data": [[1.0, 2.0, 3.0, 4.0], [1.5, 2.5, 3.5, 4.5]]
    }
}

try:
    response = requests.post(
        ENDPOINT_URL,
        headers=headers,
        json=test_payload,
        timeout=30
    )
    
    response_dict = {
        "status_code": response.status_code,
        "predictions": response.json() if response.status_code == 200 else None
    }
    
    validation = validate_model_response(response_dict, 2)
    
    print("Model Validation Results:")
    print(f"  HTTP Status: {validation['http_status']}")
    print(f"  Response Format Valid: {validation['response_format_valid']}")
    print(f"  Prediction Count Matches: {validation['prediction_count_matches']}")
    print(f"  Prediction Values Valid: {validation['prediction_values_valid']}")
    
    if validation['issues']:
        print(f"  Issues:")
        for issue in validation['issues']:
            print(f"    - {issue}")
    else:
        print(f"  ✓ All validation checks passed!")
        
except Exception as e:
    print(f"Validation failed: {e}")